In [0]:
import json

def api_service_get_indicators(symbol, limit=3):
    """
    Mô phỏng API Service (SOA Architecture)
    Truy vấn dữ liệu từ Gold Layer và trả về định dạng JSON chuẩn
    """
    try:
        # Truy vấn trực tiếp từ bảng Gold bạn đã làm ở Notebook 03
        query = f"""
            SELECT date, close, daily_return_pct, sma_20, sma_50 
            FROM workspace.stock_db.gold_indicators 
            WHERE symbol = '{symbol.upper()}' 
            ORDER BY date DESC LIMIT {limit}
        """
        df = spark.sql(query)
        rows = df.collect()
        
        if not rows:
            return json.dumps({"status": 404, "message": "Not Found"}, indent=4)
            
        # Chuyển dữ liệu sang danh sách JSON
        data_list = []
        for r in rows:
            data_list.append({
                "date": str(r.date),
                "close": r.close,
                "daily_return_pct": r.daily_return_pct,
                "sma_20": r.sma_20,
                "sma_50": r.sma_50
            })
            
        # Cấu trúc Response chuẩn API
        response = {
            "status": 200,
            "symbol": symbol.upper(),
            "data": data_list,
            "service_info": {
                "name": "Stock Data Service",
                "architecture": "SOA",
                "cloud_platform": "Databricks"
            }
        }
        return json.dumps(response, indent=4)
        
    except Exception as e:
        return json.dumps({"status": 500, "error": str(e)}, indent=4)

# ==========================================
# CHẠY ĐỂ LẤY KẾT QUẢ CHỤP ẢNH SLIDE 9
# ==========================================
print(">>> ĐANG GỌI DỊCH VỤ API (GET /api/v1/stocks/FPT.VN/indicators)")
print("-" * 50)

# Gọi hàm và in kết quả JSON
result_json = api_service_get_indicators("FPT.VN", limit=3)
print(result_json)